In [1]:
%%capture

!pip install pyspellchecker
!pip install gensim

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import random
import pandas as pd
import re

import nltk
from nltk.corpus import words, stopwords
from nltk.metrics import edit_distance
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from spellchecker import SpellChecker

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
GLOBAL_SEED = 42

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

## Preprocessing

In [5]:
def append_to_df(exist_df, text, label):
    sentences = [s.strip() for s in text.split(',') if s.strip() != ""]

    new_data = pd.DataFrame({
        'text': sentences,
        'target': label
    })

    updated_df = pd.concat([exist_df, new_data], ignore_index=True)
    return updated_df


In [6]:
data = pd.DataFrame(columns=['text', 'target'])
target_encodings = {}

for i, elem in enumerate(['Negative', 'Neutral', 'Positive']):
    with open(f'data/processed{elem}.csv') as text:
        target_encodings[elem] = i
        data = append_to_df(data, text.readline(), i)

In [7]:
data

,text,target
0,How unhappy some dogs like it though,0
1,talking to my over driver about where I'm goin...,0
2,Does anybody know if the Rand's likely to fall...,0
3,I miss going to gigs in Liverpool unhappy,0
4,There isnt a new Riverdale tonight ? unhappy,0
...,...,...
3860,Thanks for the recent follow Happy to connect ...,2
3861,- top engaged members this week happy,2
3862,ngam to weeks left for cadet pilot exam cryin...,2
3863,Great! You're welcome Josh happy ^Adam,2


In [8]:
data = data.drop_duplicates()

In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3452 entries, 0 to 3863
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    3452 non-null   object
 1   target  3452 non-null   object
dtypes: object(2)
memory usage: 80.9+ KB


In [10]:
data['target'].value_counts()

,count
target,
1,1476
2,999
0,977


In [11]:
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('words')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!


True

In [12]:
class Misspeler():
    def __init__(self, english_vocab):
        self.english_vocab = english_vocab
        self.spell_cache = {}

    def misspelings(self, word):
        if word in self.english_vocab or word.isdigit():
            return word

        if word in self.spell_cache:
            return self.spell_cache[word]

        word_len = len(word)
        prefix = word[:2]

        candidates = [
            w for w in self.english_vocab
            if w.startswith(prefix) and abs(len(w) - word_len) <= 2
        ]

        if not candidates:
            candidates = [w for w in self.english_vocab if w.startswith(word[0]) and abs(len(w) - word_len) <= 1]

        if not candidates:
            self.spell_cache[word] = word
            return word

        best_match = min(candidates, key=lambda candidate: edit_distance(word, candidate))

        if edit_distance(word, best_match) <= 2:
            self.spell_cache[word] = best_match
            return best_match

        self.spell_cache[word] = word
        return word

    def process_bunch(self, tokens):
        result = [self.misspelings(token) for token in tokens]
        return ' '.join(result)

In [13]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words('english'))

spell = SpellChecker()

In [14]:
def text_clean(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

In [15]:
def clean_and_tokenize(text, use_stop_words=False):
    cleaned = text_clean(text)
    tokens = word_tokenize(cleaned)

    if use_stop_words:
        tokens = [w for w in tokens if w not in stop_words]

    return ' '.join(tokens)

In [16]:
def stemming(text, stemmer):
    result = [stemmer.stem(token) for token in text]
    return ' '.join(result)

def lemmatization(text, lemmatizer):
    result = [lemmatizer.lemmatize(token) for token in text]
    return ' '.join(result)

def misspel(x, spell):
    res = []
    for word in x.split():
        corr = spell.correction(word)
        res.append(corr if corr else word)
    return ' '.join(res)

In [40]:
%%capture
data['clean_tokens'] = data['text'].apply(lambda x: clean_and_tokenize(x, use_stop_words=True))
data['clean_tokens_no_stop'] = data['text'].apply(lambda x: clean_and_tokenize(x, use_stop_words=False))
data['misspelings'] = data['clean_tokens_no_stop'].apply(lambda x: misspel(x, spell))

In [41]:
%%capture
data['tokens_no_stop'] = data['clean_tokens_no_stop']
data['tokens'] = data['clean_tokens']
data['stemming'] = data['clean_tokens'].apply(lambda x: stemming(x.split(), stemmer))
data['lemmatization'] = data['clean_tokens'].apply(lambda x: lemmatization(x.split(), lemmatizer))
data['stemming_misspel'] = data['misspelings'].apply(lambda x: stemming(x.split(), stemmer))
data['lematization_misspel'] = data['misspelings'].apply(lambda x: lemmatization(x.split(), lemmatizer))

In [20]:
data = data.drop_duplicates(subset=['target', 'clean_tokens', 'clean_tokens_no_stop'])

In [21]:
data.to_csv('data/processed_for_ds_1.csv', index=False)

In [22]:
data = data.fillna('missing_text')

/tmp/ipykernel_162208/648767405.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.fillna('missing_text')


In [23]:
train, test = train_test_split(data, test_size=0.2, stratify=data['target'], random_state=42)

In [24]:
def extract_features(dataframe, text_column, max_features=4096, vectorizer_type="tfidf", vectorizer=None, ngram_range=(1, 2)):
    if vectorizer:
        return vectorizer.transform(dataframe[text_column])

    if vectorizer_type == "binary":
        vectorizer = CountVectorizer(binary=True, max_features=max_features, ngram_range=ngram_range)
    elif vectorizer_type == "counts":
        vectorizer = CountVectorizer(max_features=max_features, ngram_range=ngram_range)
    elif vectorizer_type == "tfidf":
        vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)
    else:
        raise ValueError("Invalid vectorizer_type. Choose 'binary', 'counts', or 'tfidf'.")

    feature_matrix = vectorizer.fit_transform(dataframe[text_column])

    return feature_matrix, vectorizer

In [25]:
feature_mat_train = {}
feature_mat_test = {}

preprocess_columns = [
    'tokens_no_stop', 'tokens',
    'stemming', 'lemmatization',
    'stemming_misspel', 'lematization_misspel'
]

vectorizer_strategies = ['binary', 'counts', 'tfidf']

for col in preprocess_columns:
    for strategy in vectorizer_strategies:
        key_name = f"{col}_{strategy}"

        matrix_train, vec_train = extract_features(train, col, vectorizer_type=strategy)
        matrix_test = extract_features(test, col, vectorizer=vec_train)
        feature_mat_train[key_name] = matrix_train
        feature_mat_test[key_name] = matrix_test

        print(f"Generated sparse matrices for: {key_name}")


Generated sparse matrices for: tokens_no_stop_binary
Generated sparse matrices for: tokens_no_stop_counts
Generated sparse matrices for: tokens_no_stop_tfidf
Generated sparse matrices for: tokens_binary
Generated sparse matrices for: tokens_counts
Generated sparse matrices for: tokens_tfidf
Generated sparse matrices for: stemming_binary
Generated sparse matrices for: stemming_counts
Generated sparse matrices for: stemming_tfidf
Generated sparse matrices for: lemmatization_binary
Generated sparse matrices for: lemmatization_counts
Generated sparse matrices for: lemmatization_tfidf
Generated sparse matrices for: stemming_misspel_binary
Generated sparse matrices for: stemming_misspel_counts
Generated sparse matrices for: stemming_misspel_tfidf
Generated sparse matrices for: lematization_misspel_binary
Generated sparse matrices for: lematization_misspel_counts
Generated sparse matrices for: lematization_misspel_tfidf


## Most similar tweets

In [38]:
def find_most_similar_pairs(dense_matrix, original_df, top_n=10):
    sim_matrix = cosine_similarity(dense_matrix)
    upper_tri_indices = np.triu_indices(sim_matrix.shape[0], k=1)

    pair_scores = sim_matrix[upper_tri_indices]
    row_indices = upper_tri_indices[0]
    col_indices = upper_tri_indices[1]

    sorted_order = np.argsort(pair_scores)[::-1]

    for rank in range(top_n):
        match_idx = sorted_order[rank]
        score = pair_scores[match_idx]

        tweet_a_idx = row_indices[match_idx]
        tweet_b_idx = col_indices[match_idx]

        tweet_a_text = original_df.iloc[tweet_a_idx]['text']
        tweet_b_text = original_df.iloc[tweet_b_idx]['text']

        print(f"{rank + 1} Similarity Score: {score:.4f}")
        print(f" - {tweet_a_text}")
        print(f" - {tweet_b_text}")

In [39]:
for col in preprocess_columns:
    for strategy in vectorizer_strategies:
        key_name = f"{col}_{strategy}"
        print(f'\n{key_name}')
        find_most_similar_pairs(feature_mat_train[key_name], train, top_n=10)




tokens_no_stop_binary
1 Similarity Score: 1.0000
 - Good Morning Peady
 - good morning
2 Similarity Score: 1.0000
 - Hellooo happy  Xbox adHatter (XboxMadHatter)
 - Hellooo.. happy  Kyle Davies (RebelByACause)
3 Similarity Score: 1.0000
 - thanks happy
 - thanks b happy
4 Similarity Score: 1.0000
 - pake 3 unhappy
 - pake unhappy
5 Similarity Score: 1.0000
 - Hellooo happy  Xbox adHatter (XboxMadHatter)
 - Hellooo.. happy  Lil strait (Daddyy_rat)
6 Similarity Score: 1.0000
 - Hellooo happy  Xbox adHatter (XboxMadHatter)
 - Hellooo happy  Bluey (xBlueys)
7 Similarity Score: 1.0000
 - Hellooo.. happy  pussy (creepz_x)
 - Hellooo happy  PixelSavage (Pixel_Savage_Yo)
8 Similarity Score: 1.0000
 - Madam and more. Also in epaper.
 - controlling and more. Also in epaper.
9 Similarity Score: 1.0000
 - Madam and more. Also in epaper.
 - and more. Also in epaper.
10 Similarity Score: 1.0000
 - Hellooo.. happy  pussy (creepz_x)
 - Hellooo.. happy  Kyle Davies (RebelByACause)

tokens_no_stop_coun

## Models

In [28]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from gensim.models import Word2Vec
from sklearn.model_selection import GridSearchCV

In [29]:
def model_train_test(model, X_train, y_train, X_test, y_test, res):
    for col in res.index:
        for strategy in res.columns:
            key_name = f"{col}_{strategy}"

            model.fit(X_train[key_name], y_train)
            res.loc[col, strategy] = model.score(X_test[key_name], y_test)

    return res

In [30]:
y_train, y_test = train['target'], test['target']

In [31]:
linear_svc = LinearSVC(class_weight='balanced', loss='squared_hinge', random_state=42)
linear_svc_res = pd.DataFrame(columns=vectorizer_strategies, index=preprocess_columns)

linear_svc_res = model_train_test(linear_svc, feature_mat_train, y_train, feature_mat_test, y_test, linear_svc_res)

linear_svc_res

,binary,counts,tfidf
tokens_no_stop,0.872807,0.872807,0.871345
tokens,0.869883,0.872807,0.884503
stemming,0.868421,0.864035,0.888889
lemmatization,0.875731,0.874269,0.880117
stemming_misspel,0.866959,0.861111,0.868421
lematization_misspel,0.862573,0.864035,0.869883


In [32]:
log_reg = LogisticRegression(C=2.0, solver='lbfgs', random_state=42)
log_reg_res = pd.DataFrame(columns=vectorizer_strategies, index=preprocess_columns)

log_reg_res = model_train_test(log_reg, feature_mat_train, y_train, feature_mat_test, y_test, log_reg_res)

log_reg_res

,binary,counts,tfidf
tokens_no_stop,0.877193,0.880117,0.884503
tokens,0.885965,0.883041,0.884503
stemming,0.878655,0.875731,0.877193
lemmatization,0.874269,0.871345,0.885965
stemming_misspel,0.877193,0.878655,0.869883
lematization_misspel,0.878655,0.877193,0.874269


In [33]:
compelementNB = ComplementNB()
nb_res = pd.DataFrame(columns=vectorizer_strategies, index=preprocess_columns)

nb_res = model_train_test(compelementNB, feature_mat_train, y_train, feature_mat_test, y_test, nb_res)

nb_res

,binary,counts,tfidf
tokens_no_stop,0.837719,0.843567,0.840643
tokens,0.802632,0.807018,0.807018
stemming,0.828947,0.827485,0.830409
lemmatization,0.807018,0.811404,0.812865
stemming_misspel,0.858187,0.856725,0.845029
lematization_misspel,0.843567,0.846491,0.846491


In [34]:
%%capture

parameters = {'C': [0.1, 0.5, 1.0, 2.0],
              'penalty': ['l1', 'l2', 'elasticnet'],
              'solver': ['lbfgs', 'liblinear', 'sag'],
              'max_iter': [2000],
              'random_state': [42]}

log_reg = LogisticRegression(solver='liblinear', random_state=42)

clf = GridSearchCV(log_reg, parameters)
clf.fit(feature_mat_train['stemming_misspel_tfidf'], y_train)

In [35]:
cv_res = pd.DataFrame(clf.cv_results_).sort_values('rank_test_score')
cv_res['params'].iloc[0]

{'C': 2.0,
 'max_iter': 2000,
 'penalty': 'l2',
 'random_state': 42,
 'solver': 'lbfgs'}

## Bonus

In [36]:
def get_average_word2vec(tokenized_tweets, w2v_model, vector_size):
    features = []
    for tokens in tokenized_tweets:
        valid_vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]

        if valid_vectors:
            features.append(np.mean(valid_vectors, axis=0))
        else:
            features.append(np.zeros(vector_size))

    return np.array(features)

In [37]:
word2vec_linearsvc = LinearSVC(class_weight='balanced', loss='squared_hinge', random_state=42)
word2vec_linearsvc_res = pd.Series(index=preprocess_columns)

for col in preprocess_columns:
    w2v_model = Word2Vec(sentences=train[col].apply(lambda x: x.split()), vector_size=128, window=10, min_count=2, workers=4, seed=42)

    X_train_vec = get_average_word2vec(train[col].apply(lambda x: x.split()), w2v_model, 128)
    X_test_vec = get_average_word2vec(test[col].apply(lambda x: x.split()), w2v_model, 128)

    word2vec_linearsvc.fit(X_train_vec, y_train)
    word2vec_linearsvc_res.loc[col] = word2vec_linearsvc.score(X_test_vec, y_test)

word2vec_linearsvc_res

,0
tokens_no_stop,0.625731
tokens,0.758772
stemming,0.763158
lemmatization,0.769006
stemming_misspel,0.675439
lematization_misspel,0.692982
